# Use the Global Burn Log data

Everything the website shows comes from plain files on https://globalburnlog.earth/data/. No key, no login. This notebook loads the daily burning series for one area, finds its fire seasons and compares a year with the rest.

Source: NASA FIRMS active fire (MODIS Terra and Aqua, VIIRS Suomi NPP). A *burning cell-day* is an H3 resolution-5 cell with at least one MODIS detection that day, or at least two VIIRS detections. 2000-2011 values are calibrated estimates (see the Method page).

In [ ]:
import gzip, json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://globalburnlog.earth/data"

def get(path):
    raw = requests.get(f"{BASE}/{path}", timeout=60).content
    return gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw

regions = json.loads(get("regions.json"))
index = pd.DataFrame(regions["regions"])
index[["id", "name", "kind", "cells"]].head(20)

## One area's daily series

`values` is burning cell-days per day starting 2000-01-01 (empty before 2000-11-01). `lo` and `hi` are the calibration range for the MODIS-era years; they equal `values` once VIIRS measures directly (from 2012).

In [ ]:
area = "punjab-haryana"          # any id from the table above: a country code such as "ind", or a featured region
d = json.loads(get(f"series/{area}.json.gz"))
s = pd.Series([np.nan if v is None else v for v in d["values"]],
              index=pd.date_range("2000-01-01", periods=len(d["values"])), name="burning_cell_days")
lo = pd.Series([np.nan if v is None else v for v in d["lo"]], index=s.index)
hi = pd.Series([np.nan if v is None else v for v in d["hi"]], index=s.index)
s.dropna().describe()

In [ ]:
monthly = s.resample("MS").sum(min_count=28)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(monthly.index, monthly.values, lw=1.4)
ax.axvline(pd.Timestamp("2012-01-20"), ls="--", color="grey")
ax.set(title=f"Burning cell-days per month, {area}", ylabel="cell-days")
plt.show()

## Compare a year with the typical year

The site's verdict card does the same: the year's total against the other years, in standard deviations.

In [ ]:
annual = s.groupby(s.index.year).sum(min_count=360)
annual = annual.dropna()
year = 2025
others = annual.drop(year)
z = (annual[year] - others.mean()) / others.std(ddof=0)
rank = int((others > annual[year]).sum()) + 1
print(f"{year}: {annual[year]:,.0f} cell-days, {z:+.1f} standard deviations from the other years, #{rank} busiest of {len(annual)}")

## Fire season by day of year

A 3-week smoothed series, with the typical (median) year and the 10th to 90th percentile band.

In [ ]:
full = annual.index  # complete years only
doy = s[s.index.year.isin(full)]
tbl = doy.groupby([doy.index.year, doy.index.dayofyear]).sum().unstack(0).iloc[:365]
smooth = tbl.rolling(21, center=True, min_periods=10).mean()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(smooth.index, smooth.quantile(0.1, axis=1), smooth.quantile(0.9, axis=1), alpha=0.25, label="10th to 90th percentile")
ax.plot(smooth.median(axis=1), label="median year")
ax.plot(smooth[year], lw=2, label=str(year))
ax.set(xlabel="day of year", ylabel="cell-days per day"); ax.legend(); plt.show()

## Latest week and the map files

`nrt.json` is refreshed daily (near-real-time data, about 12 days behind). Map layers are binary: for each year, `map/<year>.bin.gz` holds columns of `uint32` cell index, `uint8` month and `uint8` burning days for H3 resolution-4 cells, with the cell ids in `map/cells4.bin.gz`.

In [ ]:
nrt = json.loads(get("nrt.json"))
print(nrt["start"], "to", nrt["end"])
print(nrt["areas"][area])

cells = np.frombuffer(get("map/cells4.bin.gz"), dtype="<u8")
raw = get("map/2023.bin.gz"); n = len(raw) // 6
cell = np.frombuffer(raw, "<u4", n, 0); month = np.frombuffer(raw, "u1", n, 4 * n); days = np.frombuffer(raw, "u1", n, 5 * n)
print(n, "cell-months in 2023; first cell id:", hex(int(cells[cell[0]])))